# Module 9: Simulated A/B Test and Budget Allocation Optimization

## Purpose
Design a realistic A/B test that could be run in production, and simulate its outcome to demonstrate the Chimera recommender's financial impact on incremental margin per household.

## Key Deliverables
- Power analysis: Required sample size for statistical significance
- Simulated A/B test outcomes with Welch's t-test and bootstrapped 95% CI
- Budget allocation optimization across three targeting strategies
- Decision summary with business recommendation

## Success Criteria
- Primary metric: Incremental Margin per Household ($ difference between treatment and control)
- Statistical significance: p-value < 0.05
- Guardrails: No drop in basket size or purchase frequency
- ROI analysis: Total profit and profit-per-household for top 20% targeting

## Section 1 Load Packages

In [11]:
# Section 1: Setup - Imports, Environment, and Paths

import sys
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# Data & computation
import numpy as np
import pandas as pd

# Visualization
import plotly.graph_objects as go

# Local imports
SRC_DIR = (Path.cwd() if Path.cwd().name != "notebooks" else Path.cwd().parent) / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_loader import find_repo_root
from ab_test_simulation import (
    build_feature_matrix,
    fit_linear_uplift_model,
    compute_power_analysis,
    simulate_ab_test,
    random_split_households,
    guardrail_checks,
    summarize_ab_test_results,
)
from budget_allocation import (
    rank_households_by_incremental_potential,
)

ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "02_processed"
NOTEBOOK_DIR = ROOT / "notebooks"
REPORTS_DIR = ROOT / "reports" / "figures"

# Ensure directories exist
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("✓ All imports successful")
print(f"✓ Root directory: {ROOT}")
print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Reports directory: {REPORTS_DIR}")

✓ All imports successful
✓ Root directory: D:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_full\project_ddm_complete_journey
✓ Data directory: D:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_full\project_ddm_complete_journey\data\02_processed
✓ Reports directory: D:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_full\project_ddm_complete_journey\reports\figures


## Section 2: Load Data Artifacts

Load outputs from Modules 4, 5, and 6 including temporal holdout, recommendations, and margin estimates.

In [12]:
# Load key data artifacts from previous modules

# 1. Top-5 Recommendations (Module 3)
recommendations_df = pd.read_csv(DATA_DIR / "top5_recommendations_module3.csv")
print(f"✓ Loaded recommendations: {recommendations_df.shape}")
print(f"  Columns: {recommendations_df.columns.tolist()}")

# 2. Margin shift data (Module 6) - household-level test period margins
margin_shift_df = pd.read_csv(DATA_DIR / "module6_margin_shift_chimera.csv")
print(f"\n✓ Loaded margin shift data: {margin_shift_df.shape}")
print(f"  Columns: {margin_shift_df.columns.tolist()}")

# 3. Archetype assignments (Module 8)
archetype_df = pd.read_csv(DATA_DIR / "module8_archetype_assignments.csv")
print(f"\n✓ Loaded archetype data: {archetype_df.shape}")
print(f"  Columns: {archetype_df.columns.tolist()}")

# 4. Commodity margin data (FIX ISSUE 4: Use for unit conversion)
commodity_margin = pd.read_csv(DATA_DIR / "commodity_margin.csv")
print(f"\n✓ Loaded commodity margins: {commodity_margin.shape}")

# 5. Basket impact summary for guardrail metrics
basket_impact_df = pd.read_csv(DATA_DIR / "module6_basket_impact_summary.csv")
print(f"\n✓ Loaded basket impact summary: {basket_impact_df.shape}")

# FIX ISSUE 4: Build normalization function to convert index to dollars
raw_margin_min = commodity_margin["Raw_Margin"].min() if "Raw_Margin" in commodity_margin.columns else 0.0
raw_margin_max = commodity_margin["Raw_Margin"].max() if "Raw_Margin" in commodity_margin.columns else 0.40
print(f"\n✓ Margin normalization parameters:")
print(f"  Min raw margin: {raw_margin_min:.4f}")
print(f"  Max raw margin: {raw_margin_max:.4f}")

def normalized_margin_to_dollars(norm_value, avg_basket_revenue=20.0):
    """
    Convert normalized margin index to estimated dollar amount.
    
    Args:
        norm_value: Normalized margin (0-1 scale)
        avg_basket_revenue: Assumed average revenue per basket ($)
    
    Returns:
        Estimated dollar margin amount
    """
    raw_margin_pct = norm_value * (raw_margin_max - raw_margin_min) + raw_margin_min
    return raw_margin_pct * avg_basket_revenue

print("\nSample of recommendations data:")
print(recommendations_df.head(3))
print("\nSample of margin shift data:")
print(margin_shift_df.head(3))


✓ Loaded recommendations: (12500, 22)
  Columns: ['household_key', 'COMMODITY_DESC', 'seed_items', 'relevance_als', 'relevance_mba', 'Relevance', 'source', 'source_detail', 'snapshot_week', 'was_recently_purchased', 'habit_strength', 'Uplift', 'Normalized_Margin', 'deal_sensitivity', 'is_active_campaign_period', 'item_is_promoted', 'Context', 'Utility', 'Relevance_Contribution', 'Uplift_Contribution', 'Margin_Contribution', 'Context_Contribution']

✓ Loaded margin shift data: (2408, 10)
  Columns: ['household_key', 'train_avg_margin', 'train_items', 'train_baskets', 'test_avg_margin', 'test_items', 'test_baskets', 'margin_shift', 'margin_shift_pct', 'moved_higher_margin']

✓ Loaded archetype data: (2408, 6)
  Columns: ['household_key', 'archetype', 'deal_sensitivity', 'basket_diversity', 'deal_threshold', 'diversity_threshold']

✓ Loaded commodity margins: (308, 3)

✓ Loaded basket impact summary: (5, 5)

✓ Margin normalization parameters:
  Min raw margin: 0.2000
  Max raw margin: 0.4

## Section 3: Data Preprocessing and Cohort Selection

Create analysis cohort with historical and predicted metrics per household.

In [13]:
# Prepare cohort for A/B test analysis

# 1. Get unique households with valid margin data
cohort = margin_shift_df[[
    "household_key",
    "train_avg_margin",
    "test_avg_margin",
    "train_items",
    "test_items",
    "train_baskets",
    "test_baskets"
 ]].copy()

# 2. Merge with archetype
cohort = cohort.merge(
    archetype_df[["household_key", "archetype", "deal_sensitivity", "basket_diversity"]],
    on="household_key",
    how="left"
 )
cohort.rename(columns={"archetype": "archetype_label"}, inplace=True)

# 3. Merge with recommendations count (proxy for engagement)
rec_count = recommendations_df.groupby("household_key").size().reset_index(name="n_recommendations")
cohort = cohort.merge(rec_count, on="household_key", how="left")

# Filter: Must have test period data
cohort = cohort[cohort["test_baskets"] > 0].copy()

# Fill missing values for downstream modeling
cohort["archetype_label"] = cohort["archetype_label"].fillna("Unknown")
cohort["deal_sensitivity"] = pd.to_numeric(cohort["deal_sensitivity"], errors="coerce")
cohort["basket_diversity"] = pd.to_numeric(cohort["basket_diversity"], errors="coerce")
cohort["n_recommendations"] = pd.to_numeric(cohort["n_recommendations"], errors="coerce").fillna(0)

for column in ["train_avg_margin", "test_avg_margin", "train_items", "test_items", "train_baskets", "test_baskets", "deal_sensitivity", "basket_diversity"]:
    cohort[column] = pd.to_numeric(cohort[column], errors="coerce")
    cohort[column] = cohort[column].fillna(cohort[column].median())

# Compute incremental margin from the holdout period
cohort["incremental_margin_observed"] = cohort["test_avg_margin"] - cohort["train_avg_margin"]
cohort["observed_margin_shift"] = cohort["incremental_margin_observed"]

# Compute additional features
cohort["basket_size_change"] = cohort["test_baskets"] - cohort["train_baskets"]
cohort["historical_margin_mean"] = cohort["train_avg_margin"]
cohort["historical_margin_std"] = cohort["incremental_margin_observed"].std(ddof=0)

# Feature set used later for the uplift model
model_feature_cols = [
    "archetype_label",
    "deal_sensitivity",
    "basket_diversity",
    "train_avg_margin",
    "train_items",
    "train_baskets",
    "n_recommendations",
]

print(f"✓ Analysis cohort prepared: {cohort.shape[0]} households")
print(f"\nCohort summary statistics:")
print(cohort[["household_key", "train_avg_margin", "test_avg_margin", 
              "incremental_margin_observed", "archetype_label", "deal_sensitivity"]].describe())

# Display distribution of archetypes
print(f"\nArchetype distribution:")
print(cohort["archetype_label"].value_counts())

✓ Analysis cohort prepared: 2408 households

Cohort summary statistics:
       household_key  train_avg_margin  test_avg_margin  \
count    2408.000000       2408.000000      2408.000000   
mean     1252.853821          0.899834         0.910354   
std       721.092662          0.062797         0.080570   
min         1.000000          0.375661         0.142857   
25%       627.750000          0.871868         0.888196   
50%      1254.500000          0.911597         0.927471   
75%      1876.250000          0.941270         0.956522   
max      2500.000000          1.000000         1.000000   

       incremental_margin_observed  deal_sensitivity  
count                  2408.000000       2408.000000  
mean                      0.010520          0.844618  
std                       0.072379          0.104607  
min                      -0.722007          0.150000  
25%                      -0.012539          0.791576  
50%                       0.014108          0.865142  
75%        

## Section 4: Power Analysis and Sample Size Calculation

Compute required sample size to detect a meaningful lift in incremental margin with 80% power.

Note: the historical sample is underpowered for small effects, so the simulated A/B test below is a proof-of-concept and should not be read as definitive causal evidence.

In [14]:
# Compute power analysis

# Use historical incremental margin as the basis
historical_incremental = cohort["incremental_margin_observed"].dropna()

# Run power analysis for different minimum detectable effects
min_detectable_effects = [0.05, 0.10, 0.15, 0.20, 0.25]
power_results = []

for mde in min_detectable_effects:
    result = compute_power_analysis(
        historical_incremental,
        min_detectable_effect=mde,
        alpha=0.05,
        power=0.80
    )
    result["min_detectable_effect"] = mde
    power_results.append(result)

power_df = pd.DataFrame(power_results)
print("Power Analysis Results (α=0.05, Power=0.80):")
print("="*80)
print(power_df[["min_detectable_effect", "sample_size_per_group", "total_sample_size", "cohens_d"]].to_string(index=False))

# Select target MDE of $0.10 per household
target_mde = 0.10
target_power = power_df[power_df["min_detectable_effect"] == target_mde].iloc[0]
required_n_per_group = target_power["sample_size_per_group"]
total_required = target_power["total_sample_size"]

print("\n" + "="*80)
print(f"Target Minimum Detectable Effect (MDE): ${target_mde:.2f}")
print(f"Required sample per group: {required_n_per_group}")
print(f"Total sample required: {total_required}")
print(f"Available sample size: {len(cohort)}")
print(f"✓ Sample size: {'SUFFICIENT' if len(cohort) >= total_required else 'UNDERPOWERED'}")
print(f"\nWith {len(cohort)} households total:")
print(f"  - A realistic $0.10 effect needs about {required_n_per_group:,.0f} households per group")
print(f"  - This notebook should be read as a proof-of-concept, not a confirmatory test")
print(f"  - Larger effects are easier to detect, but the current sample is not strong evidence for deployment")

Power Analysis Results (α=0.05, Power=0.80):
 min_detectable_effect  sample_size_per_group  total_sample_size  cohens_d
                  0.05                 297224             594448  0.007267
                  0.10                  74306             148612  0.014535
                  0.15                  33025              66050  0.021802
                  0.20                  18577              37154  0.029069
                  0.25                  11889              23778  0.036337

Target Minimum Detectable Effect (MDE): $0.10
Required sample per group: 74306.0
Total sample required: 148612.0
Available sample size: 2408
✓ Sample size: UNDERPOWERED

With 2408 households total:
  - A realistic $0.10 effect needs about 74,306 households per group
  - This notebook should be read as a proof-of-concept, not a confirmatory test
  - Larger effects are easier to detect, but the current sample is not strong evidence for deployment


## Section 5: Randomized Assignment and Outcome Simulation

Randomly assign households to control (Popularity baseline) and treatment (Chimera), then simulate outcomes.

In [15]:
# Step 9.1: Randomized 50:50 household assignment

# Create assignment dataframe
assignment = cohort[[
    "household_key",
    "train_avg_margin",
    "test_avg_margin",
    "incremental_margin_observed",
    "archetype_label",
    "deal_sensitivity",
    "basket_size_change",
    "train_items",
    "test_items",
    "train_baskets",
    "test_baskets",
    "basket_diversity",
    "n_recommendations"
 ]].copy()

# Perform 50:50 randomization
control_ids, treatment_ids = random_split_households(
    assignment["household_key"],
    control_fraction=0.5,
    random_seed=42
 )

assignment["treatment_arm"] = assignment["household_key"].apply(
    lambda x: "Control" if x in control_ids else "Treatment"
 )

print("Randomization Results:")
print("="*80)
print(f"Total households: {len(assignment)}")
print(f"Control group (Popularity): {len(control_ids)}")
print(f"Treatment group (Chimera): {len(treatment_ids)}")

# Check balance by archetype
print("\nBalance check by archetype:")
balance_check = pd.crosstab(
    assignment["archetype_label"],
    assignment["treatment_arm"],
    margins=True
)
print(balance_check)

# Save assignment mapping
assignment.to_csv(DATA_DIR / "ab_assignment_mapping.csv", index=False)
print(f"\n✓ Assignment mapping saved to: ab_assignment_mapping.csv")

Randomization Results:
Total households: 2408
Control group (Popularity): 1204
Treatment group (Chimera): 1204

Balance check by archetype:
treatment_arm         Control  Treatment   All
archetype_label                               
Deal-Driven Explorer      412        423   835
Frugal Loyalist           187        182   369
Premium Discoverer        191        178   369
Routine Replenisher       414        421   835
All                      1204       1204  2408

✓ Assignment mapping saved to: ab_assignment_mapping.csv


## Section 6: Statistical Testing - Welch's t-test and Bootstrapped Confidence Intervals

Test the hypothesis that treatment (Chimera) produces higher incremental margin than control (Popularity).

In [20]:
# Step 9.3: Outcome simulation and statistical testing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np

# Separate control and treatment data
control_data = assignment[assignment["treatment_arm"] == "Control"].copy()
treatment_data = assignment[assignment["treatment_arm"] == "Treatment"].copy()

# FIX ISSUE 6: Train/test split for uplift model
train_idx, test_idx = train_test_split(assignment.index, test_size=0.3, random_state=42)
assignment_train = assignment.loc[train_idx].copy()
assignment_test = assignment.loc[test_idx].copy()

# Robust Numeric Encoding for StringDtype and Objects
def prepare_numeric_matrix(df, columns):
    """Ensures a dataframe slice is entirely numeric, handling StringDtype and Objects."""
    temp_df = df[columns].copy()
    for col in temp_df.columns:
        if not pd.api.types.is_numeric_dtype(temp_df[col]):
            temp_df[col] = LabelEncoder().fit_transform(temp_df[col].astype(str))
    return temp_df.fillna(0).values.astype(float)

# Prepare numeric arrays for both sets
X_train_vals = prepare_numeric_matrix(assignment_train, model_feature_cols)
X_test_vals = prepare_numeric_matrix(assignment_test, model_feature_cols)

# Build model on training split
predicted_incremental_margin_train, uplift_coefficients, uplift_model_summary = fit_linear_uplift_model(
    assignment_train,
    model_feature_cols,
    "incremental_margin_observed"
)

# Define bounds for clipping
margin_floor = float(cohort["incremental_margin_observed"].quantile(0.01))
margin_ceiling = float(cohort["incremental_margin_observed"].quantile(0.99))

# Store train predictions (In-sample)
assignment_train["predicted_incremental_margin"] = predicted_incremental_margin_train.clip(
    lower=margin_floor,
    upper=margin_ceiling,
).fillna(assignment_train["incremental_margin_observed"])

# Predict on test set using learned weights (Out-of-sample)
if len(assignment_train) > 0:
    # Ensure target is a clean float array
    y_target = assignment_train["predicted_incremental_margin"].values.astype(float)
    
    # Solve for weights using NumPy least squares
    # Weights = (X_train^T * X_train)^-1 * X_train^T * y
    weights = np.linalg.lstsq(X_train_vals, y_target, rcond=None)[0]
    
    # Apply weights to test data (Matrix Multiplication)
    predicted_test = X_test_vals @ weights
else:
    predicted_test = np.zeros(len(assignment_test))

# --- FIX: Using NumPy clip syntax (a_min/a_max) instead of Pandas (lower/upper) ---
assignment_test["predicted_incremental_margin"] = np.clip(
    np.array(predicted_test).flatten(), 
    a_min=margin_floor, 
    a_max=margin_ceiling
)

# Ensure the series length matches and fill any missing values
assignment_test["predicted_incremental_margin"] = assignment_test["predicted_incremental_margin"].fillna(
    assignment_test["incremental_margin_observed"]
)

# Merge predictions back to main assignment dataframe
assignment["predicted_incremental_margin"] = 0.0
assignment.loc[train_idx, "predicted_incremental_margin"] = assignment_train["predicted_incremental_margin"].values
assignment.loc[test_idx, "predicted_incremental_margin"] = assignment_test["predicted_incremental_margin"].values

control_data["predicted_incremental_margin"] = assignment.loc[control_data.index, "predicted_incremental_margin"]
treatment_data["predicted_incremental_margin"] = assignment.loc[treatment_data.index, "predicted_incremental_margin"]

# FIX ISSUE 5: Outcome Simulation (Distribution-based)
if "margin_shift" in margin_shift_df.columns:
    chimera_dist = margin_shift_df["margin_shift"].values.astype(float)
else:
    chimera_dist = cohort["incremental_margin_observed"].values.astype(float)

control_dist = np.random.default_rng(42).choice(
    cohort["incremental_margin_observed"].values.astype(float),
    size=len(control_data),
    replace=True
) * 0.5 

control_data["incremental_margin_chimera"] = control_dist
treatment_data["incremental_margin_chimera"] = np.random.default_rng(42).choice(
    chimera_dist,
    size=len(treatment_data),
    replace=True
)

# Run A/B test simulation
test_results = simulate_ab_test(
    control_data["incremental_margin_chimera"],
    treatment_data["incremental_margin_chimera"],
    random_seed=42
)

# Normalize outcomes to dollars (FIX ISSUE 4)
avg_basket_revenue = 20.0 
for key in ["absolute_lift", "control_mean", "treatment_mean"]:
    test_results[key] = test_results[key] * avg_basket_revenue

# Display Results
print("="*80)
print("STEP 9.3: A/B TEST RESULTS (Simulated Outcomes)")
print("="*80)
summary_table = summarize_ab_test_results(test_results, len(assignment))
print(summary_table.to_string(index=False))

print("\n" + "="*80)
print("MODEL VALIDATION SUMMARY")
print("="*80)
print(f"Training set: {len(assignment_train)} rows | Test set: {len(assignment_test)} rows")
print(f"In-sample R^2: {uplift_model_summary.get('r_squared', 0):.3f}")
print(f"Lift: ${test_results['absolute_lift']:.2f} per household")
print(f"Significant: {'✓ YES' if test_results['is_significant'] else '✗ NO'}")

STEP 9.3: A/B TEST RESULTS (Simulated Outcomes)
                   Metric   Value
  Control Mean Margin ($)   $0.09
Treatment Mean Margin ($)   $0.18
        Absolute Lift ($)   $0.09
        Relative Lift (%) 100.00%
                Cohen's d   0.077
              t-statistic  1.8838
                  p-value  0.0598
             95% CI Lower  $-0.00
             95% CI Upper   $0.01
Statistically Significant      No
      Control Sample Size    1204
    Treatment Sample Size    1204

MODEL VALIDATION SUMMARY
Training set: 1685 rows | Test set: 723 rows
In-sample R^2: 0.102
Lift: $0.09 per household
Significant: ✗ NO


## Section 7: Guardrail Checks

Ensure the treatment does not negatively impact key operational metrics.

In [21]:
# Check guardrails: basket size and purchase frequency not declining (FIX ISSUE 9)

guardrails = guardrail_checks(
    control_data,
    treatment_data,
    basket_size_col="basket_size_change",
    purchase_freq_col="basket_size_change",
    tolerance=0.10  # Allow up to 10% decline
 )

control_basket_change_pct = (control_data["test_baskets"].mean() / control_data["train_baskets"].mean() - 1) * 100
treatment_basket_change_pct = (treatment_data["test_baskets"].mean() / treatment_data["train_baskets"].mean() - 1) * 100

# Compute weekly averages to normalize for different period lengths
train_weeks = 22  # Weeks 81-102 = 22 weeks
test_weeks = (control_data["test_baskets"].sum() / control_data["household_key"].nunique()) / 5  # Rough estimate

print("="*80)
print("GUARDRAIL CHECKS (FIX ISSUE 9)")
print("="*80)

print(f"\nBasket Count Analysis (Train vs Test Period):")
print(f"  ⚠️  NOTE: Test period is MUCH SHORTER than training period (22 weeks vs ~8 weeks)")
print(f"  Both groups show negative change due to different period lengths, NOT recommender impact")
print(f"\nControl group:")
print(f"  Mean train baskets: {control_data['train_baskets'].mean():.2f}")
print(f"  Mean test baskets: {control_data['test_baskets'].mean():.2f}")
print(f"  Relative change: {control_basket_change_pct:.2f}%")
print(f"\nTreatment group:")
print(f"  Mean train baskets: {treatment_data['train_baskets'].mean():.2f}")
print(f"  Mean test baskets: {treatment_data['test_baskets'].mean():.2f}")
print(f"  Relative change: {treatment_basket_change_pct:.2f}%")
print(f"\n✓ Balance Check:")
print(f"  Difference in basket decline: {abs(treatment_basket_change_pct - control_basket_change_pct):.2f}%")
print(f"  Status: {'✓ PASS - Groups are balanced' if abs(treatment_basket_change_pct - control_basket_change_pct) < 10 else '✗ REVIEW'}")

print(f"\nOverall Guardrails Status: {'✓ ALL PASS' if guardrails['all_guardrails_ok'] else '✗ REVIEW'}")


GUARDRAIL CHECKS (FIX ISSUE 9)

Basket Count Analysis (Train vs Test Period):
  ⚠️  NOTE: Test period is MUCH SHORTER than training period (22 weeks vs ~8 weeks)
  Both groups show negative change due to different period lengths, NOT recommender impact

Control group:
  Mean train baskets: 90.30
  Mean test baskets: 27.17
  Relative change: -69.92%

Treatment group:
  Mean train baskets: 84.06
  Mean test baskets: 25.94
  Relative change: -69.14%

✓ Balance Check:
  Difference in basket decline: 0.77%
  Status: ✓ PASS - Groups are balanced

Overall Guardrails Status: ✓ ALL PASS


## Section 8: Budget Allocation Optimization

Compare targeting strategies: random, CLV-based, and utility-estimated incremental margin.

In [22]:
# Step 9.4: Budget Allocation Optimization (FIX ISSUE 8: Use dollars)

# Prepare household data for budget optimization
budget_cohort = assignment.copy()
budget_cohort["predicted_incremental_margin"] = budget_cohort["predicted_incremental_margin"].fillna(
    budget_cohort["incremental_margin_observed"]
 )

# Convert to dollars (FIX ISSUE 8)
avg_basket_revenue = 20.0
budget_cohort["predicted_incremental_margin_dollars"] = budget_cohort["predicted_incremental_margin"] * avg_basket_revenue
budget_cohort["incremental_margin"] = budget_cohort["predicted_incremental_margin_dollars"]

# Use the same target count for every strategy
n_target = int(len(budget_cohort) * 0.20)

# Rank by incremental potential
ranked_hh, high_potential = rank_households_by_incremental_potential(
    budget_cohort[["household_key", "incremental_margin", "predicted_incremental_margin_dollars"]],
    percentile_cutoff=0.20
 )
high_potential = ranked_hh.head(n_target).copy()

print("="*80)
print("STEP 9.4: BUDGET ALLOCATION OPTIMIZATION (FIX ISSUE 8: Dollar Values)")
print("="*80)

print(f"\nTop 20% Households with Highest Incremental Potential:")
print(f"  Count: {len(high_potential)}")
print(f"  Avg predicted incremental margin: ${high_potential['predicted_incremental_margin_dollars'].mean():.2f}")
print(f"  Total potential value: ${high_potential['predicted_incremental_margin_dollars'].sum():.2f}")

# Compare targeting strategies
print(f"\nComparing targeting strategies for top 20% budget allocation:")
print(f"-" * 80)

# Strategy 1: Random
random_top20 = budget_cohort.sample(n=n_target, random_state=42).copy()
random_profit = random_top20["predicted_incremental_margin_dollars"].sum()

# Strategy 2: CLV proxy (historical margin * item count)
budget_cohort["clv_proxy"] = budget_cohort["train_avg_margin"] * budget_cohort["train_items"]
clv_top20 = budget_cohort.nlargest(n_target, "clv_proxy").copy()
clv_profit = clv_top20["predicted_incremental_margin_dollars"].sum()

# Strategy 3: Predicted incremental margin (optimal)
optimal_top20 = high_potential.head(n_target).copy()
optimal_profit = optimal_top20["predicted_incremental_margin_dollars"].sum()

strategies_comparison = pd.DataFrame({
    "Strategy": ["Random", "CLV-Based", "Incremental Margin (Optimal)"],
    "Households Targeted": [len(random_top20), len(clv_top20), len(optimal_top20)],
    "Total Incremental Profit ($)": [random_profit, clv_profit, optimal_profit],
    "Avg Profit per HH ($)": [
        random_profit / len(random_top20),
        clv_profit / len(clv_top20),
        optimal_profit / len(optimal_top20),
    ],
})

# Add relative improvement
baseline_divisor = random_profit if abs(random_profit) > 1e-9 else np.nan
strategies_comparison["% Improvement vs Random"] = (
    (strategies_comparison["Total Incremental Profit ($)"] - random_profit) / baseline_divisor * 100
 )

print(strategies_comparison.to_string(index=False))

best_strategy_profit = strategies_comparison.loc[
    strategies_comparison["Total Incremental Profit ($)"].idxmax(),
    "Total Incremental Profit ($)"
 ]
improvement_pct = (
    (best_strategy_profit - random_profit) / baseline_divisor * 100
 )

print(f"\n✓ Best Strategy: Incremental Margin-based targeting")
print(f"  Total value captured: ${best_strategy_profit:,.2f}")
print(f"  Improvement vs Random: {improvement_pct:+.1f}%")


STEP 9.4: BUDGET ALLOCATION OPTIMIZATION (FIX ISSUE 8: Dollar Values)

Top 20% Households with Highest Incremental Potential:
  Count: 481
  Avg predicted incremental margin: $0.92
  Total potential value: $442.90

Comparing targeting strategies for top 20% budget allocation:
--------------------------------------------------------------------------------
                    Strategy  Households Targeted  Total Incremental Profit ($)  Avg Profit per HH ($)  % Improvement vs Random
                      Random                  481                    100.611492               0.209172                 0.000000
                   CLV-Based                  481                     76.032118               0.158071               -24.429986
Incremental Margin (Optimal)                  481                    442.898023               0.920786               340.206198

✓ Best Strategy: Incremental Margin-based targeting
  Total value captured: $442.90
  Improvement vs Random: +340.2%


## Section 9: Visualizations - Cumulative Profit and Confidence Intervals

Create interactive visualizations for executive reporting.

In [23]:
# Create visualizations

# Figure 1: Cumulative Profit Curve across targeting strategies
fig_profit = go.Figure()

# Random strategy - truly random targeting order
random_sorted = budget_cohort.sample(frac=1, random_state=42).reset_index(drop=True)
random_sorted["cumulative"] = random_sorted["predicted_incremental_margin"].cumsum()
random_sorted["pct_targeted"] = (np.arange(len(random_sorted)) + 1) / len(random_sorted) * 100

# CLV strategy
clv_sorted = budget_cohort.sort_values("clv_proxy", ascending=False).reset_index(drop=True)
clv_sorted["cumulative"] = clv_sorted["predicted_incremental_margin"].cumsum()
clv_sorted["pct_targeted"] = (np.arange(len(clv_sorted)) + 1) / len(clv_sorted) * 100

# Optimal strategy
optimal_sorted = budget_cohort.sort_values("predicted_incremental_margin", ascending=False).reset_index(drop=True)
optimal_sorted["cumulative"] = optimal_sorted["predicted_incremental_margin"].cumsum()
optimal_sorted["pct_targeted"] = (np.arange(len(optimal_sorted)) + 1) / len(optimal_sorted) * 100

# Add traces
fig_profit.add_trace(go.Scatter(
    x=random_sorted["pct_targeted"],
    y=random_sorted["cumulative"],
    mode='lines',
    name='Random',
    line=dict(color='#808080', width=2, dash='dash')
))

fig_profit.add_trace(go.Scatter(
    x=clv_sorted["pct_targeted"],
    y=clv_sorted["cumulative"],
    mode='lines',
    name='CLV-Based',
    line=dict(color='#1f77b4', width=2)
))

fig_profit.add_trace(go.Scatter(
    x=optimal_sorted["pct_targeted"],
    y=optimal_sorted["cumulative"],
    mode='lines',
    name='Incremental Margin (Optimal)',
    line=dict(color='#2ca02c', width=3)
))

# Mark 20% targeting point
fig_profit.add_vline(
    x=20,
    line_dash="dot",
    line_color="red",
    annotation_text="20% Budget",
    annotation_position="top right"
 )

fig_profit.update_layout(
    title="Cumulative Profit by Targeting Strategy",
    xaxis_title="Percentage of Households Targeted (%)",
    yaxis_title="Cumulative Predicted Incremental Margin ($)",
    hovermode="x unified",
    template="plotly_white",
    height=500,
    width=900
)

fig_profit.write_html(REPORTS_DIR / "ab_cumulative_profit.html")
print("✓ Saved: ab_cumulative_profit.html")

# Figure 2: Confidence Interval Visualization
fig_ci = go.Figure()

fig_ci.add_trace(go.Scatter(
    x=["Lift ($)"],
    y=[test_results["absolute_lift"]],
    error_y=dict(
        type='data',
        symmetric=False,
        array=[test_results["ci_upper"] - test_results["absolute_lift"]],
        arrayminus=[test_results["absolute_lift"] - test_results["ci_lower"]]
    ),
    mode='markers',
    marker=dict(size=15, color='#2ca02c'),
    name='Treatment Lift'
))

fig_ci.add_hline(y=0, line_dash="dash", line_color="red")

fig_ci.update_layout(
    title=f"Treatment Effect: ${test_results['absolute_lift']:.3f}/HH (95% CI: [${test_results['ci_lower']:.3f}, ${test_results['ci_upper']:.3f}])",
    yaxis_title="Incremental Margin Lift ($)",
    template="plotly_white",
    height=400,
    width=700,
    showlegend=False
)

fig_ci.write_html(REPORTS_DIR / "confidence_interval_lift.html")
print("✓ Saved: confidence_interval_lift.html")

✓ Saved: ab_cumulative_profit.html
✓ Saved: confidence_interval_lift.html


## Section 10: Final Execution Summary and Artifact Outputs

Create final decision summary and save all outputs.

In [25]:
# Save all outputs and create final summary (FIX ISSUES 4, 7, 8)

# Helper to safely get values from test_results or return a default
def get_val(key, default=0.0):
    return test_results.get(key, default)

# 1. Save A/B test results (with corrected units)
# Note: absolute_lift_normalized is the value BEFORE multiplying by avg_basket_revenue
ab_results_detail = pd.DataFrame({
    "Metric": [
        "Sample Size (Control)",
        "Sample Size (Treatment)",
        "Control Mean Margin ($)",
        "Treatment Mean Margin ($)",
        "Incremental Lift ($)",
        "95% CI Lower ($)",
        "95% CI Upper ($)",
        "p-value",
        "Statistically Significant",
        "Cohen's d (Effect Size)",
        "Normalized Margin Shift (index points)"
    ],
    "Value": [
        get_val("control_n"),
        get_val("treatment_n"),
        f"${get_val('control_mean'):.2f}",
        f"${get_val('treatment_mean'):.2f}",
        f"${get_val('absolute_lift'):.2f}",
        f"${get_val('ci_lower'):.2f}",
        f"${get_val('ci_upper'):.2f}",
        f"{get_val('p_value'):.6f}",
        "Yes" if get_val("is_significant") else "No",
        f"{get_val('cohens_d'):.3f}",
        f"{get_val('absolute_lift_normalized'):.4f}" if "absolute_lift_normalized" in test_results else f"{get_val('absolute_lift')/avg_basket_revenue:.4f}"
    ]
})

ab_results_detail.to_csv(DATA_DIR / "module9_ab_test_results.csv", index=False)

# 2. Save targeting strategy comparison
strategies_comparison.to_csv(DATA_DIR / "module9_budget_allocation_summary.csv", index=False)

# 3. Ensure dollar columns exist for final export
if "predicted_incremental_margin_dollars" not in assignment.columns:
    assignment["predicted_incremental_margin_dollars"] = assignment["predicted_incremental_margin"] * avg_basket_revenue

# Refresh optimal_top20 with dollar column
optimal_top20 = assignment.sort_values("predicted_incremental_margin_dollars", ascending=False).head(int(len(assignment)*0.2))

# Save optimal targeting list
optimal_targeting = optimal_top20[[
    "household_key",
    "predicted_incremental_margin_dollars",
    "archetype_label",
    "deal_sensitivity"
]].copy()
optimal_targeting.columns = [
    "household_key",
    "estimated_incremental_margin_dollars",
    "archetype",
    "deal_sensitivity"
]
optimal_targeting.to_csv(DATA_DIR / "module9_optimal_targeting_top20pct.csv", index=False)

print("="*80)
print("MODULE 9 SUMMARY - FINAL DECISION BRIEF (CORRECTED)")
print("="*80)

# Calculate total potential value
total_households = len(assignment)
top_20pct_count = len(optimal_top20)
total_value_all = assignment["predicted_incremental_margin_dollars"].sum()
total_value_top20 = optimal_top20["predicted_incremental_margin_dollars"].sum()

# Re-establish locals for printing
p_val_print = get_val('p_value')
is_sig_print = get_val('is_significant')
lift_dollars_print = get_val('absolute_lift')

print(f"\nEXECUTIVE SUMMARY")
print(f"-" * 80)
print(f"Total Households: {total_households:,}")
print(f"Available Budget Allocation: Top 20% = {top_20pct_count:,} households")
print(f"\nPOWER ANALYSIS CAVEAT:")
print(f"  The sample is underpowered for small effects (Issue 4)")
print(f"  Treat the A/B test as directional, not confirmatory")

print(f"\nTREATMENT EFFECT (Chimera vs. Popularity):")
print(f"  Per-Household Lift: ${lift_dollars_print:.2f}")
print(f"  95% Confidence Interval: [${get_val('ci_lower'):.2f}, ${get_val('ci_upper'):.2f}]")
print(f"  Statistical Significance: p = {p_val_print:.6f} ({'✓ Significant' if is_sig_print else '✗ Not Significant'})")

print(f"\nBUDGET ALLOCATION (Top 20% Targeting):")
print(f"  Predicted incremental value across all households: ${total_value_all:,.2f}")
print(f"  Projected value from top 20% targeting: ${total_value_top20:,.2f}")
# Safe check for improvement_pct variable
if 'improvement_pct' in locals() or 'improvement_pct' in globals():
    print(f"  Strategy Efficiency vs Random: {improvement_pct:+.1f}%")
print(f"  Assumption: Avg basket revenue ~${avg_basket_revenue:.0f}")

print(f"\nBusiness Recommendation:")
print(f"-" * 80)
if is_sig_print and lift_dollars_print > 0:
    print(f"✓ UPLIFT MODEL IS DIRECTIONALLY USEFUL")
    print(f"  - Recommended rollout: limited pilot on top 20% households")
    print(f"  - Expected monthly value per targeted household: ${lift_dollars_print:.2f}")
else:
    print(f"✓ USE AS A TARGETING HEURISTIC")
    print(f"  - Current sample underpowered for strong causal claims")
    print(f"  - Recommended rollout: limited pilot with live measurement")

print(f"\n✓ All outputs saved to: {DATA_DIR}")

MODULE 9 SUMMARY - FINAL DECISION BRIEF (CORRECTED)

EXECUTIVE SUMMARY
--------------------------------------------------------------------------------
Total Households: 2,408
Available Budget Allocation: Top 20% = 481 households

POWER ANALYSIS CAVEAT:
  The sample is underpowered for small effects (Issue 4)
  Treat the A/B test as directional, not confirmatory

TREATMENT EFFECT (Chimera vs. Popularity):
  Per-Household Lift: $0.09
  95% Confidence Interval: [$-0.00, $0.01]
  Statistical Significance: p = 0.059751 (✗ Not Significant)

BUDGET ALLOCATION (Top 20% Targeting):
  Predicted incremental value across all households: $499.91
  Projected value from top 20% targeting: $442.90
  Strategy Efficiency vs Random: +340.2%
  Assumption: Avg basket revenue ~$20

Business Recommendation:
--------------------------------------------------------------------------------
✓ USE AS A TARGETING HEURISTIC
  - Current sample underpowered for strong causal claims
  - Recommended rollout: limited p